In [1]:
import matplotlib.pyplot as plt
import time
import numpy as np
import time
from joblib import Parallel, delayed
from threadpoolctl import threadpool_limits
import pickle
import ot

In [3]:
import numpy as np, time, ot, os
from concurrent.futures import ProcessPoolExecutor, as_completed
from threadpoolctl import threadpool_limits

def get_entropy_np(pi):
    return (pi * np.log(np.clip(pi, 1e-12, None))).sum()

def sinkhorn_trials_chunk(mu1, mu2, sig1, sig2, N, reg, Tchunk, seed,
                          use_log=False, need_indep=False, return_times=False):
    """
    Returns a 6-tuple of numpy arrays (length = Tchunk):
      (cost_xy, cost_xx, cost_yy_same, cost_yy_indep_or_None,
       sink_div_emp, sink_div_indep_or_None)
    If return_times=True, returns an additional 4-tuple of timing arrays:
      (time_xy, time_xx, time_yy_same, time_yy_indep_or_None)
    """
    method = 'sinkhorn_log' if use_log else 'sinkhorn'
    with threadpool_limits(limits=1):  # 1 BLAS/OpenMP thread per process
        rng = np.random.default_rng(seed)
        a = np.full(N, 1.0/N)

        cost_xy       = np.empty(Tchunk, dtype=np.float64)
        cost_xx       = np.empty(Tchunk, dtype=np.float64)
        cost_yy_same  = np.empty(Tchunk, dtype=np.float64)
        cost_yy_indep = np.empty(Tchunk, dtype=np.float64) if need_indep else None
        cost_yy_same2  = np.empty(Tchunk, dtype=np.float64) if need_indep else None

        sink_div_emp   = np.empty(Tchunk, dtype=np.float64)
        sink_div_self = np.empty(Tchunk, dtype=np.float64) if need_indep else None

        ent_xy       = np.empty(Tchunk, dtype=np.float64)
        ent_xx       = np.empty(Tchunk, dtype=np.float64)
        ent_yy_same  = np.empty(Tchunk, dtype=np.float64)
        ent_yy_indep = np.empty(Tchunk, dtype=np.float64) if need_indep else None
        ent_yy_same2  = np.empty(Tchunk, dtype=np.float64) if need_indep else None
        
        time_xy = time_xx = time_yy_same = time_yy_ind = None
        if return_times:
            time_xy      = np.empty(Tchunk, dtype=np.float64)
            time_xx      = np.empty(Tchunk, dtype=np.float64)
            time_yy_same = np.empty(Tchunk, dtype=np.float64)
            if need_indep:
                time_yy_ind = np.empty(Tchunk, dtype=np.float64)
                time_yy_same2 = np.empty(Tchunk, dtype=np.float64)

        for t in range(Tchunk):
            # draw the pair
            x1 = rng.standard_normal(N)[:, None] * sig1 + mu1
            x2 = rng.standard_normal(N)[:, None] * sig2 + mu2

            # XY cost
            M = ot.dist(x1, x2, metric='sqeuclidean')
            t0 = time.time()
            pi = ot.sinkhorn(a, a, M, reg,
                                numItermax=10000, stopThr=1e-9, method=method)
            c_xy = np.sum(M*pi)
            if return_times: time_xy[t] = time.time() - t0
            cost_xy[t] = float(c_xy)
            ent_xy[t] = get_entropy_np(pi)

            # XX self-cost (same sample)
            Mx = ot.dist(x1, x1, metric='sqeuclidean')
            t0 = time.time()
            pi = ot.sinkhorn(a, a, Mx, reg,
                                numItermax=10000, stopThr=1e-9, method=method)
            c_xx = np.sum(Mx*pi)
            if return_times: time_xx[t] = time.time() - t0
            cost_xx[t] = float(c_xx)
            ent_xx[t] = get_entropy_np(pi)

            # YY self-cost (same sample)
            My = ot.dist(x2, x2, metric='sqeuclidean')
            t0 = time.time()
            pi = ot.sinkhorn(a, a, My, reg,
                                     numItermax=10000, stopThr=1e-9, method=method)
            c_yy_same = np.sum(My*pi)
            if return_times: time_yy_same[t] = time.time() - t0
            cost_yy_same[t] = float(c_yy_same)
            ent_yy_same[t] = get_entropy_np(pi)

            # Empirical Sinkhorn divergence (same-sample self terms)
            sink_div_emp[t] = cost_xy[t] - 0.5*(cost_xx[t] + cost_yy_same[t])

            # Optional: YY self-cost from independent draws (population-like)
            if need_indep:
                x2_2 = rng.standard_normal(N)[:, None] * sig2 + mu2
                Myi = ot.dist(x2, x2_2, metric='sqeuclidean')
                t0 = time.time()
                pi = ot.sinkhorn(a, a, Myi, reg,
                                        numItermax=10000, stopThr=1e-9, method=method)
                c_yy_ind = np.sum(Myi*pi)
                if return_times: time_yy_ind[t] = time.time() - t0
                cost_yy_indep[t]  = float(c_yy_ind)
                ent_yy_indep[t] = get_entropy_np(pi)
                
                My2 = ot.dist(x2_2, x2_2, metric='sqeuclidean')
                t0 = time.time()
                pi = ot.sinkhorn(a, a, My2, reg,
                                        numItermax=10000, stopThr=1e-9, method=method)
                c_yy_same2 = np.sum(My2*pi)
                if return_times: time_yy_same2[t] = time.time() - t0
                cost_yy_same2[t]  = float(c_yy_same2)
                ent_yy_same2[t] = get_entropy_np(pi)
                sink_div_self[t] = cost_yy_indep[t] - 0.5*(cost_yy_same[t] + cost_yy_same2[t])

        ret = (cost_xy, cost_xx, cost_yy_same, cost_yy_indep, cost_yy_same2, 
               ent_xy, ent_xx, ent_yy_same, ent_yy_indep, ent_yy_same2, 
               sink_div_emp, sink_div_self)
        if return_times:
            ret += (time_xy, time_xx, time_yy_same, time_yy_ind if need_indep else None, time_yy_same2 if need_indep else None)
        return ret

def calibrate_trial_time(N_vals, sig2, reg, mu1=0.0, mu2=0.0, sig1=1.0, reps=10):
    times = {}
    for N in N_vals:
        rng = np.random.default_rng(0)
        a = np.full(N, 1.0/N)
        # warm-up once
        x1 = rng.standard_normal(N)[:, None]*sig1 + mu1
        x2 = rng.standard_normal(N)[:, None]*sig2 + mu2
        M  = ot.dist(x1, x2, metric='sqeuclidean')
        ot.sinkhorn2(a, a, M, reg)
        # measure
        t0 = time.time()
        for _ in range(reps):
            x1 = rng.standard_normal(N)[:, None]*sig1 + mu1
            x2 = rng.standard_normal(N)[:, None]*sig2 + mu2
            M  = ot.dist(x1, x2, metric='sqeuclidean')
            ot.sinkhorn2(a, a, M, reg)
        times[N] = (time.time()-t0)/reps
    return times

def run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=40, target_chunk_sec=1.0, seed=123, use_log=False,
          need_indep=True, return_times=True):
    """
    Parallel, adaptive, dynamic scheduler for many Sinkhorn trials.

    Returns a dict with keys:
      - 'cost_xy', 'cost_xx', 'cost_yy_same', ['cost_yy_indep']
      - 'sink_div_emp', ['sink_div_indep']
      - optionally time arrays: 'time_xy', 'time_xx', 'time_yy_same', ['time_yy_indep']
      - 't_per' (per-N calibrated single-trial times)

    Shapes: (len(N_set), len(sig2_set), T) for all arrays.
    """

    sig2_list = list(sig2_set)
    N_list = list(N_set)
    nN, nS = len(N_list), len(sig2_list)

    # 1) Calibrate per-N time (use a representative sigma)
    rep_sig2 = sig2_list[nS // 2]
    t_per = calibrate_trial_time(N_list, sig2=rep_sig2, reg=reg)

    # 2) Build adaptive chunks
    # chunk_size(N) ~ target_chunk_sec / t_per[N]
    tasks = []  # (i, j, start, end, N, s2, seed)
    rng = np.random.default_rng(seed)
    for i, N in enumerate(N_list):
        per_trial = max(t_per[N], 1e-4)
        chunk_size = max(1, int(target_chunk_sec / per_trial))
        for j, s2 in enumerate(sig2_list):
            remaining = T
            start = 0
            while remaining > 0:
                c = min(chunk_size, remaining)
                end = start + c
                tasks.append((i, j, start, end, N, s2, int(rng.integers(0, 2**32 - 1))))
                start = end
                remaining -= c

    # 3) LPT-ish: do "heavier" (larger N, larger chunk) first
    tasks.sort(key=lambda x: (-x[4], -(x[3]-x[2])))

    # 4) Preallocate outputs
    shape = (nN, nS, T)
    Cxy  = np.empty(shape, dtype=np.float64)
    Cxx  = np.empty(shape, dtype=np.float64)
    CyyS = np.empty(shape, dtype=np.float64)
    CyyI = np.empty(shape, dtype=np.float64) if need_indep else None
    CyyS2 = np.empty(shape, dtype=np.float64) if need_indep else None

    Exy  = np.empty(shape, dtype=np.float64)
    Exx  = np.empty(shape, dtype=np.float64)
    EyyS = np.empty(shape, dtype=np.float64)
    EyyI = np.empty(shape, dtype=np.float64) if need_indep else None
    EyyS2 = np.empty(shape, dtype=np.float64) if need_indep else None
    
    S_emp = np.empty(shape, dtype=np.float64)
    S_self = np.empty(shape, dtype=np.float64) if need_indep else None

    Txy = Txx = TyyS = TyyI = None
    if return_times:
        Txy  = np.empty(shape, dtype=np.float64)
        Txx  = np.empty(shape, dtype=np.float64)
        TyyS = np.empty(shape, dtype=np.float64)
        if need_indep:
            TyyI = np.empty(shape, dtype=np.float64)
            TyyS2 = np.empty(shape, dtype=np.float64)

    # 5) Submit and collect dynamically
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        future_to_meta = {}
        for (i, j, start, end, N, s2, s) in tasks:
            fut = ex.submit(
                sinkhorn_trials_chunk,
                mu1, mu2, sig1, s2, N, reg, end - start, s,
                use_log=use_log, need_indep=need_indep, return_times=return_times
            )
            future_to_meta[fut] = (i, j, start, end)

        for fut in as_completed(future_to_meta):
            i, j, start, end = future_to_meta[fut]
            res = fut.result()

            # Unpack results from sinkhorn_trials_chunk
            (cost_xy, cost_xx, cost_yy_same, cost_yy_indep, cost_yy_same2, 
             ent_xy, ent_xx, ent_yy_same, ent_yy_indep, ent_yy_same2, 
             sink_emp, sink_self, *times) = res

            Cxy[i, j, start:end]  = cost_xy
            Cxx[i, j, start:end]  = cost_xx
            CyyS[i, j, start:end] = cost_yy_same
            Exy[i, j, start:end]  = ent_xy
            Exx[i, j, start:end]  = ent_xx
            EyyS[i, j, start:end] = ent_yy_same
            S_emp[i, j, start:end] = sink_emp

            if need_indep:
                CyyI[i, j, start:end]  = cost_yy_indep
                CyyS2[i, j, start:end] = cost_yy_same2
                EyyI[i, j, start:end]  = ent_yy_indep
                EyyS2[i, j, start:end] = ent_yy_same2
                S_self[i, j, start:end] = sink_self

            if return_times:
                time_xy, time_xx, time_yy_same, time_yy_ind, time_yy_same2 = times
                Txy[i, j, start:end]  = time_xy
                Txx[i, j, start:end]  = time_xx
                TyyS[i, j, start:end] = time_yy_same
                if need_indep:
                    TyyI[i, j, start:end] = time_yy_ind
                    TyyS2[i, j, start:end] = time_yy_same2

    out = {
        "cost_xy": Cxy,
        "cost_xx": Cxx,
        "cost_yy_same": CyyS,
        "ent_xy": Exy,
        "ent_xx": Exx,
        "ent_yy_same": EyyS,
        "sink_div_emp": S_emp,
        "t_per": t_per,
    }
    if need_indep:
        out["cost_yy_indep"] = CyyI
        out["cost_yy_same2"] = CyyS2
        out["ent_yy_indep"] = EyyI
        out["ent_yy_same2"] = EyyS2
        out["sink_div_self"] = S_self
    if return_times:
        out["time_xy"] = Txy
        out["time_xx"] = Txx
        out["time_yy_same"] = TyyS
        if need_indep:
            out["time_yy_indep"] = TyyI
            out["time_yy_same2"] = TyyS2

    return out


In [4]:
mu1=0
mu2=0
sig1=1
N_set=[10, 100]
sig2_set=np.linspace(0.01,1.2,120)
T = 10000

max_workers=40
target_chunk_sec=1.0
seed=42

In [5]:
reg=100
start = time.time()
results = run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=max_workers, target_chunk_sec=target_chunk_sec, seed=seed, use_log=False)
print(time.time() - start)
filename = f'./entreg_results/results_eps{reg}_N10_100.pickle'
with open(filename, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

1898.645403623581


In [6]:
reg=20
start = time.time()
results = run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=max_workers, target_chunk_sec=target_chunk_sec, seed=seed, use_log=False)
print(time.time() - start)
filename = f'./entreg_results/results_eps{reg}_N10_100.pickle'
with open(filename, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

1975.9334301948547


In [7]:
reg=10
start = time.time()
results = run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=max_workers, target_chunk_sec=target_chunk_sec, seed=seed, use_log=False)
print(time.time() - start)
filename = f'./entreg_results/results_eps{reg}_N10_100.pickle'
with open(filename, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

1993.01922249794


In [8]:
reg=5
start = time.time()
results = run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=max_workers, target_chunk_sec=target_chunk_sec, seed=seed, use_log=False)
print(time.time() - start)
filename = f'./entreg_results/results_eps{reg}_N10_100.pickle'
with open(filename, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

2090.1063075065613


In [9]:
reg=1
start = time.time()
results = run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=max_workers, target_chunk_sec=target_chunk_sec, seed=seed, use_log=False)
print(time.time() - start)
filename = f'./entreg_results/results_eps{reg}_N10_100.pickle'
with open(filename, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(
/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(
/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(
/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(
/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sin

5185.47918844223


In [ ]:
reg=0.2
start = time.time()
results = run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=max_workers, target_chunk_sec=target_chunk_sec, seed=seed, use_log=False)
print(time.time() - start)
filename = f'./entreg_results/results_eps{reg}_N10_100.pickle'
with open(filename, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(
/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(
/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(
/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(
/usr/local/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sin

In [ ]:
reg=2
start = time.time()
results = run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=max_workers, target_chunk_sec=target_chunk_sec, seed=seed, use_log=False)
print(time.time() - start)
filename = f'./entreg_results/results_eps{reg}_N10_100.pickle'
with open(filename, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
reg=0.5
start = time.time()
results = run_T(mu1, mu2, sig1, sig2_set, N_set, reg, T,
          max_workers=max_workers, target_chunk_sec=target_chunk_sec, seed=seed, use_log=False)
print(time.time() - start)
filename = f'./entreg_results/results_eps{reg}_N10_100.pickle'
with open(filename, 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)